In [ ]:
#Итоговый проект Python
#Схема проекта:
#API → data → validate_data(data) → new_data → PostgreSQL

In [ ]:
#0.Подготовка окружения и настройка логирования

In [ ]:
# Установка библиотеки для работы с PostgreSQL
%pip install psycopg2-binary

In [ ]:
# Установка библиотеки для работы с Google Sheets
%pip install gspread

In [ ]:
#установка библиотеки для безопасного хранения пароля
%pip install python-dotenv

In [ ]:
#импортируем необходимые библиотеки
import os
import logging
import requests
from datetime import datetime, timedelta, timezone
import ast
import psycopg2
import gspread

#Создаём имя журнала с текущей датой
log_filename = datetime.now().strftime('%Y-%m-%d') + '.log'

#Настраиваем запись событий в лог-файл
logging.basicConfig(
    filename=log_filename,
    format='%(asctime)s %(name)s %(levelname)s: %(message)s',
    level=logging.INFO,
    encoding='utf-8',
    filemode='a',
    force=True
)

#Удаляем журналы старше периода хранения
def cleanup_logs():
    today = datetime.now().date()
    cutoff_date = today - timedelta(days=2)
    for filename in os.listdir():
        if filename.endswith('.log'):
            date_str = filename[:-4]

            try:
                file_date = datetime.strptime(date_str, '%Y-%m-%d')
            except ValueError:
                continue

            if file_date.date() < cutoff_date:
                os.remove(filename)
                logging.info(f"Удалён устаревший лог-файл: {filename}")

cleanup_logs()                

In [ ]:
#Настройка переменных окружения
#Загрузка ключа API и пароля PostgreSQL из локального файла .env. 
#Конфиденциальные данные хранятся отдельно от исходного кода и не публикуются в репозитории.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

client_key = os.getenv("ITRESUME_CLIENT_KEY")
db_password = os.getenv("POSTGRES_PASSWORD")

In [ ]:
# 1. Получение данных из API
#Создаю функцию fetch_data(), принимающую начальную и конечную даты периода.
#Внутри функции формирую параметры запроса: данные клиента и период получения статистики.
#Записываю в лог начало загрузки данных.
#С помощью requests.get() отправляю GET-запрос к API, ограничивая время ожидания.
#Проверяю HTTP-статус ответа с помощью raise_for_status().
#Преобразую JSON-ответ в Python-объект с помощью response.json().
#Вывожу тип полученных данных и количество записей.
#При успешной загрузке записываю в лог сообщение уровня INFO с количеством записей.
#С помощью try/except перехватываю ошибки запроса, записываю их в лог
#с указанием status_code (если он доступен) и повторно вызываю исключение через raise.
#Возвращаю полученные данные для дальнейшей обработки.

In [ ]:
api_url = "https://api.itresume.ru/statistics/"

# Настройки периода выгрузки
START_DATE = "2023-05-31 00:00:00"
END_DATE = "2023-06-01 00:00:00"

def fetch_data(start_date, end_date):

    params = {
        "client": "Skillfactory",
        "client_key": client_key,
        "start": start_date,
        "end": end_date
    }

    logging.info("Начало загрузки данных из API")

    try:
        response = requests.get(api_url, params=params, timeout=60)
        response.raise_for_status()

        # Преобразование ответа API в Python-объект
        data = response.json()
        
        # Проверка типа полученных данных и количества записей
        if not isinstance(data, list):
            logging.error("Ошибка API: ответ не является списком записей")
            raise ValueError("Ответ API должен быть списком")
        print(type(data), len(data))

        logging.info(f"Загрузка данных из API завершена успешно. Получено записей: {len(data)}")
        
        return data
        
    except requests.exceptions.RequestException as e:
        status_code = e.response.status_code if e.response is not None else None
        logging.error(f"Ошибка при запросе к API. status_code: {status_code}. Ошибка: {e}")
        raise

In [ ]:
# Получение статистики за выбранный период
data = fetch_data(START_DATE, END_DATE)

In [ ]:
#2 Изучение структуры одной записи
#Перед обработкой всего списка изучаю структуру одной записи `data[0]`.
#Часть нужных полей находится непосредственно в записи, но ещё три поля
#содержатся внутри `passback_params`.

In [ ]:
data[0]


In [ ]:
#Проверяю тип `passback_params`, чтобы понять, как извлечь из него данные.

In [ ]:
type(data[0]['passback_params'])


In [ ]:
# Преобразование `passback_params` из строки в словарь
#`passback_params` приходит из API как строка (`str`), хотя по своей структуре
#содержит данные в виде Python-словаря.
#С помощью `ast.literal_eval()` безопасно преобразую эту строку в словарь,
#чтобы получить доступ к отдельным значениям по ключам.
#После преобразования проверяю тип результата.

In [ ]:
passback = ast.literal_eval(data[0]['passback_params'])
type(passback)

In [ ]:
### Проверка содержимого `passback_params`
#После преобразования `passback_params` в словарь проверяю,
#что могу обратиться к нужным значениям по ключам.
#Из `passback_params` для будущей таблицы нужны три поля:
#- `oauth_consumer_key`
#- `lis_result_sourcedid`
#- `lis_outcome_service_url`

In [ ]:
print(passback['oauth_consumer_key'])
print(passback['lis_result_sourcedid'])
print(passback['lis_outcome_service_url'])

In [ ]:
## Формирование записи для будущей таблицы
#Объединяю данные из основной записи API и из преобразованного `passback_params` в один словарь.
#Названия ключей соответствуют полям, которые в дальнейшем будут записываться в таблицу PostgreSQL.
#Пока выполняю преобразование только для первой записи `data[0]`,чтобы проверить логику перед обработкой всего списка.

In [ ]:
test_dict = {'user_id' : data[0]['lti_user_id'], 'oauth_consumer_key' : passback['oauth_consumer_key'], 'lis_result_sourcedid' : passback['lis_result_sourcedid'], 'lis_outcome_service_url' : passback['lis_outcome_service_url'], 'is_correct' : data[0]['is_correct'],  'attempt_type' : data[0]['attempt_type'], 'created_at':data[0]['created_at']}

In [ ]:
test_dict


In [ ]:
### 3. Валидация: Реализация функции обработки данных
# Создаю функцию, которая последовательно перебирает полученные из API записи,
# преобразует passback_params, формирует словарь с нужными полями,
# проверяет корректность данных и добавляет прошедшие проверку записи в новый список.

In [ ]:
def validate_data(data):
    new_data = []
    logging.info(f"Начало валидации данных. Получено записей: {len(data)}")
    for record in data:
        try:
            passback = ast.literal_eval(record['passback_params'])
            if not isinstance(passback, dict):
                logging.warning("Пропущена запись. passback_params не является словарём")
                continue
            if record['is_correct'] == 1:
                is_correct = True
            elif record['is_correct'] == 0:
                is_correct = False
            elif record['is_correct'] is None:
                is_correct = None
            else:
                logging.warning("Пропущена запись. Некорректное значение is_correct")
                continue

            new_record = {
                'user_id': record['lti_user_id'],
                'oauth_consumer_key': passback.get('oauth_consumer_key'),
                'lis_result_sourcedid': passback.get('lis_result_sourcedid'),
                'lis_outcome_service_url': passback.get('lis_outcome_service_url'),
                'is_correct': is_correct,
                'attempt_type': record['attempt_type'],
                'created_at': record['created_at']
            }

            if (
                new_record['attempt_type'] not in ('run', 'submit')
                or not isinstance(new_record['user_id'], str) or new_record['user_id'] == ''
                or (not isinstance(new_record['lis_result_sourcedid'], str) and new_record['lis_result_sourcedid'] is not None) or new_record['lis_result_sourcedid'] == ''
                or (not isinstance(new_record['lis_outcome_service_url'], str) and new_record['lis_outcome_service_url'] is not None) or new_record['lis_outcome_service_url'] == ''
                or (not isinstance(new_record['oauth_consumer_key'], str) and new_record['oauth_consumer_key'] is not None)
            ):
                logging.warning("Пропущена запись. Некорректные значения полей new_record")
                continue
        except KeyError as e:
            logging.warning(f"Пропущена запись. Отсутствует ключ: {e}")
            continue
        except SyntaxError:
            logging.warning("Пропущена запись. Некорректный passback_params")
            continue
        except (ValueError, TypeError):
            logging.warning("Пропущена запись. Некорректный passback_params")
            continue
            
        #проверка согласованности attempt_type и  is_correct   
        if (
            (new_record['attempt_type'] == 'run' and new_record['is_correct'] is not None)
            or (new_record['attempt_type'] == 'submit' and new_record['is_correct'] is None)
        ):
            logging.warning("Пропущена запись. Несогласованные attempt_type и is_correct")
            continue  
            
        try:
            datetime.strptime(new_record['created_at'], '%Y-%m-%d %H:%M:%S.%f')
        except ValueError:
            logging.warning("Пропущена запись. Некорректная дата")
            continue

        new_data.append(new_record)
    logging.info(
        f"Валидация завершена. Получено: {len(data)}, "
        f"Корректных: {len(new_data)}, "
        f"Пропущено: {len(data) - len(new_data)}"
    )    
    return new_data

In [ ]:
test_data = [
    {
        'lti_user_id': 'user_1',
        'passback_params': "{'oauth_consumer_key': '', 'lis_result_sourcedid': 'course_1', 'lis_outcome_service_url': 'https://example.com'}",
        'is_correct': None,
        'attempt_type': 'run',
        'created_at': '2023-05-31 00:00:28.249578'
    },
    {
        'lti_user_id': 'user_2',
        'passback_params': "{'oauth_consumer_key': '', 'lis_result_sourcedid': 'course_1', 'lis_outcome_service_url': 'https://example.com'}",
        'is_correct': None,
        'attempt_type': 'banana',
        'created_at': '2023-05-31 00:00:28.249578'
    }
]

In [ ]:
test_result= validate_data(test_data)

In [ ]:
test_result

In [ ]:
# Обработка данных API: валидация и подсчёт корректных записей

In [ ]:
clean_data = validate_data(data)

len(clean_data)

In [ ]:
### Проверка типов данных после валидации
#Убеждаемся, что поле is_correct содержит только значения True, False или None, 
#а числа 0 и 1 преобразованы в булев тип.

In [ ]:
{(record['is_correct'], type(record['is_correct'])) for record in clean_data}

In [ ]:
## 4. PostgreSQL и загрузка подготовленных данных в базу

In [ ]:
logging.info("Начало подключения к PostgreSQL")

try:
    conn = psycopg2.connect(
        dbname='final_project',
        user='postgres',
        password=db_password,
        host='localhost'
    )
    logging.info("Подключение к PostgreSQL выполнено успешно")

except Exception as e:
    logging.error(f"Ошибка подключения к PostgreSQL: {e}")
    raise

In [ ]:
#создаем курсор

In [ ]:
cursor = conn.cursor()

In [ ]:
#создаем таблицу

In [ ]:
try:
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS attempts (
            user_id TEXT,
            oauth_consumer_key TEXT,
            lis_result_sourcedid TEXT,
            lis_outcome_service_url TEXT,
            is_correct BOOLEAN,
            attempt_type TEXT,
            created_at TIMESTAMP,
            CONSTRAINT unique_user_attempt UNIQUE (user_id, created_at)
        )
    """)
    conn.commit()
    logging.info("Таблица attempts подготовлена")
except Exception as e:
    conn.rollback()
    logging.error(f"Ошибка подготовки таблицы attempts: {e}")
    raise

In [ ]:
query = """
INSERT INTO attempts (
    user_id,
    oauth_consumer_key,
    lis_result_sourcedid,
    lis_outcome_service_url,
    is_correct,
    attempt_type,
    created_at
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (user_id, created_at) DO NOTHING
"""

In [ ]:
#загружаем проверенные записи в PostgreSQL, пропуская дубликаты

In [ ]:
logging.info(f"Начало загрузки {len(clean_data)} записей в PostgreSQL")

inserted_count = 0
try:
    for record in clean_data:
        values = (
            record['user_id'],
            record['oauth_consumer_key'],
            record['lis_result_sourcedid'],
            record['lis_outcome_service_url'],
            record['is_correct'],
            record['attempt_type'],
            record['created_at'],
        )

        cursor.execute(query, values)
        inserted_count += cursor.rowcount

    conn.commit()
    logging.info(
        f"Загрузка в PostgreSQL завершена. "
        f"Обработано записей: {len(clean_data)}, "
        f"Добавлено новых: {inserted_count}"
    )

except Exception as e:
    conn.rollback()
    logging.error(f"Ошибка загрузки данных в PostgreSQL: {e}")
    raise

In [ ]:
#Проверка результатов загрузки данных в PostgreSQL

In [ ]:
cursor.execute("SELECT COUNT(*) FROM attempts")
print("Записей в PostgreSQL:", cursor.fetchone()[0])

In [ ]:
cursor.close()
conn.close()

logging.info("Соединение с PostgreSQL закрыто")

In [ ]:
#5. Агрегация данных за день в Google Sheets

In [ ]:
def agregate_daily_data(day):
    daily_stats = {}
    day = datetime.strptime(day, '%Y-%m-%d').date()
    
    for record in clean_data:
        date = datetime.strptime(record['created_at'], '%Y-%m-%d %H:%M:%S.%f').date()
        if date==day:
            if date not in daily_stats:
                daily_stats[date] = {'total_attempts': 0, 'successful_attempts': 0, 'unique_users': set()}
            daily_stats[date]['total_attempts'] += 1
            if record['is_correct'] is True:
                daily_stats[date]['successful_attempts'] += 1
            daily_stats[date]['unique_users'].add(record['user_id'])
            
    if day not in daily_stats:
        return None
        
    res={'total_attempts':daily_stats[day]['total_attempts'], 'successful_attempts':daily_stats[day]['successful_attempts'], 'unique_users':len(daily_stats[day]['unique_users'])}
      
    return res

In [ ]:
def  upload_to_sheets(row):
    headers = [
    'Дата',
    'Всего попыток',
    'Успешных попыток',
    'Уникальных пользователей'
    ]
    try:
        gc = gspread.service_account(filename='orbital-signal-439813-i3-db2a6f06a623.json')
        spreadsheet = gc.open('Daily Statistics')
        worksheet = spreadsheet.sheet1
        if worksheet.row_values(1) == []:
            worksheet.update(range_name='A1', values=[headers])
            logging.info('Заголовки Google-таблицы успешно созданы')
        if row[0] in worksheet.col_values(1):
            logging.warning("Статистика за эту дату уже загружена в таблицу.")
        else:
            worksheet.append_row(row)
            logging.info('Данные успешно отправлены в Google Sheets')
    except Exception as e:
        logging.error(f'Ошибка отправки данных в Google Sheets: {e}')
        raise

In [ ]:
#получаем статистику и отправляем ее в Google Sheets

In [ ]:
target_date = '2023-05-31'
stats = agregate_daily_data(target_date)

if stats is not None:
    row = [
        target_date,
        stats['total_attempts'],
        stats['successful_attempts'],
        stats['unique_users']
    ]
    upload_to_sheets(row)
else:
    logging.warning('Данные отсутствуют')

In [ ]:
#6. Настройка оповещений на почту

In [ ]:
import getpass

sender_email = 'python.analytics.project@mail.ru'
password = getpass.getpass('Введите пароль приложения: ')

In [ ]:
from email.message import EmailMessage

In [ ]:
import smtplib
import ssl
import certifi

In [ ]:
def send_email(sender_email, password, subject, message):
    #Создаём письмо с переданными темой и текстом
    msg = EmailMessage()
    msg.set_content(message)
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = sender_email
    
    #Настраиваем защищённое SSL-соединение.
    smtp_server = 'smtp.mail.ru'
    port = 465
    context = ssl.create_default_context(cafile=certifi.where())

    try:
        #Python подключается к Mail.ru и авторизуется.
        with smtplib.SMTP_SSL(smtp_server, port, context=context) as server:
            server.login(sender_email, password)
            #Письмо отправляется.
            server.send_message(msg=msg)
            logging.info('Email-уведомление успешно отправлено')

    except Exception as e:
        logging.error(f'Ошибка отправки email-уведомления: {e}')
        raise

In [ ]:
subject = 'ETL: обработка данных завершена'
message = 'Данные успешно обработаны и загружены.'

send_email(sender_email, password, subject, message)